# Download new target

If starting up a new target download it to the bulkdock directory `$BULK/TARGETS`, using the UI below.

In [1]:
# imports
from fragalysis.widgets import download
from pathlib import Path
import shutil
from os import environ
import json

In [2]:
# this is the destination you should download to:
bulk_targets_dir = Path(environ["BULK"]) / "TARGETS"
print(bulk_targets_dir)

/opt/xchem-fragalysis-2/bemery/BulkDock/TARGETS


In [3]:
# start the download widget
download()

╭─────────────────╮
│ download target │
╰─────────────────╯

# Initialise Directories for New Cycle

In [ ]:
target_name = "Template" # change this to match target name in in Fragalysis
cycle_number = 1

In [ ]:
target_dir = Path() / target_name.lower()
cycle_dir = target_dir / f"cycle_{cycle_number:02}"
fragmenstein_dir = cycle_dir / "fragmenstein"
knitwork_dir = cycle_dir / "knitwork"
knitwork_pure_output_dir = knitwork_dir / "knitwork_pure_output"
knitwork_impure_output_dir = knitwork_dir / "knitwork_impure_output"
gnina_dir = cycle_dir / "gnina"
gnina_inputs_dir = gnina_dir / "inputs"
gnina_outputs_dir = gnina_dir / "outputs"

In [ ]:
# create directories
cycle_dir.mkdir(parents=True, exist_ok=True)
fragmenstein_dir.mkdir(parents=True, exist_ok=True)
gnina_dir.mkdir(parents=True, exist_ok=True)
gnina_inputs_dir.mkdir(parents=True, exist_ok=True)
gnina_outputs_dir.mkdir(parents=True, exist_ok=True)
knitwork_dir.mkdir(parents=True, exist_ok=True)
knitwork_pure_output_dir.mkdir(parents=True, exist_ok=True)
knitwork_impure_output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# copy aligned files from $BULK to new directory
shutil.copytree(bulk_targets_dir / target_name / "aligned_files", target_dir / "aligned_files")

# Set Up Bulkdock / HIPPO

In [ ]:
%load_ext autoreload
%autoreload 2
import hippo
import mrich
from mrich import print
from pathlib import Path
from os import environ
import shutil
import molparse as mp
import plotly.express as px

### Assumes Bulkdock is configured

#### __In terminal :__

```
    cd $BULK

    python -m bulkdock setup <FRAGALYSIS_DOWNLOAD_DIR_NAME> (just the target name, not the full path)

    This creates an SQLite in $BULK/TARGETS/<FRAGALYSIS_DOWNLOAD_DIR_NAME>
```

In [ ]:
target_dir = Path(environ["BULK"]) / "TARGETS" / target_name

In [ ]:
animal = hippo.HIPPO(target_name, target_dir / f"{target_name}.sqlite")

# Create Fragmenstein and Knitwork Inputs

In [4]:
animal.tags

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 animal.tags                                                                                  │
│   2                                                                                              │
│                                                                                                  │
│ ╭─────────────────────────────────────────── locals ───────────────────────────────────────────╮ │
│ │ bulk_targets_dir = PosixPath('/opt/xchem-fragalysis-2/bemery/BulkDock/TARGETS')              │ │
│ │          environ = environ({                                                                 │ │
│ │                    │   'HYDRA_LAUNCHER_EXTRA_ARGS': '--external-launcher',                   │ │
│ │                    │   'MAMBA_ALWAYS_YES': 'yes',                                            │ │
│ │                    │   'CONDA_SHLVL': '1',                                                   │ │
│ │                    │   'SLURM_MEM_PER_CPU': '2500',                                          │ │
│ │                    │   'LS_COLORS':                                                          │ │
│ │                    'rs=0:di=38;5;33:ln=38;5;51:mh=00:pi=40;38;5;11:so=38;5;13:do=38;5;5:bd=… │ │
│ │                    │   'LD_LIBRARY_PATH':                                                    │ │
│ │                    '/usr/local/cuda/compat:/opt/xchem-fragalysis-2/bemery/conda/lib:/dls_sw… │ │
│ │                    │   'CONDA_EXE': '/opt/xchem-fragalysis-2/bemery/conda/bin/conda',        │ │
│ │                    │   'SLURM_NODEID': '0',                                                  │ │
│ │                    │   'SLURM_TASK_PID': '3430298',                                          │ │
│ │                    │   'SSH_CONNECTION': '172.23.162.11 47910 172.23.176.6 22',              │ │
│ │                    │   ... +107                                                              │ │
│ │                    })                                                                        │ │
│ │             exit = <IPython.core.autocall.ZMQExitAutocall object at 0x7fb9801712a0>          │ │
│ │      get_ipython = <bound method InteractiveShell.get_ipython of                             │ │
│ │                    <ipykernel.zmqshell.ZMQInteractiveShell object at 0x7fb9801707c0>>        │ │
│ │               In = [                                                                         │ │
│ │                    │   '',                                                                   │ │
│ │                    │   '# imports\nfrom fragalysis.widgets import download\nfrom pathlib     │ │
│ │                    import Path\nimpor'+43,                                                   │ │
│ │                    │   '# this is the destination you should download to:\nbulk_targets_dir  │ │
│ │                    = Path(enviro'+46,                                                        │ │
│ │                    │   '# start the download widget\ndownload()',                            │ │
│ │                    │   'animal.tags'                                                         │ │
│ │                    ]                                                                         │ │
│ │             json = <module 'json' from                                                       │ │
│ │                    '/opt/xchem-fragalysis-2/bemery/conda/envs/xchem-fff/lib/python3.10/json… │ │
│ │              Out = {}                                                                        │ │
│ │             quit = <IPython.core.autocall.ZMQExitAutocall object at 0x7fb9801712a0>          │ │
│ │           shutil = <module 'shutil' from                 

In [ ]:
fragment_hits = animal.poses(tag="[Other] FragScn") # change to your merge tag of choice
fragment_hits.write_sdf(cycle_dir / "hits.sdf")

In [ ]:
# Check how many tagged compounds will be processed
fragment_hits

In [ ]:
# download reference apo PDB for Fragmenstein - need a fragmenstein directory
ref_pose = fragment_hits[0]
shutil.copy(ref_pose.delig_path, cycle_dir / "fragmenstein")

# Run Fragmenstein

#### __In terminal :__

```
cd fragmenstein

sbatch --job-name "fragmenstein" --mem 16000 /opt/xchem-fragalysis-2/bemery/slurm/run_bash_with_xchem-fragmenstein_conda.sh /opt/xchem-fragalysis-2/bemery/slurm/run_fragmenstein.sh

```

# Run Knitwork on Squonk


Presumes Knitwork is configured.

__in terminal__
```
python -m knitwork fragment fffbe/<target_name>/hits.sdf --output-dir fffbe/<target_name>/fragment_output

python -m knitwork pure-merge
python -m knitwork impure-merge
```

pure merge maintains the core structure of the original fragments
impure merge can make small core modifications to the original fragments

# Redock Scaffolds with BulkDock

## Generate BulkDock inputs

#### __In Terminal:__

Fragmenstein scaffolds:
```
cd fragmenstein

python /opt/xchem-fragalysis-2/bemery/slurm/fragmenstein_to_bulkdock.py

# edit target name below
cp fragmenstein_bulkdock_input.csv $BULK/INPUTS/<Target_Name>_fragmenstein.csv
```

Knitwork scaffolds:

File structure should be : 
```
/knitwork
    /knitwork_pure_output
        /<target_name>_pure_merges.sdf
    /knitwork_impure_output
        /<target_name>_impure_merges.sdf
```
```
cd ../knitwork

python /opt/xchem-fragalysis-2/bemery/slurm/knitwork_SDF_to_bulkdock.py /opt/xchem-fragalysis-2/bemery/fffbe/<target_name>/<cycle_XX>/knitwork/

cp knitwork_pure_bulkdock_input.csv $BULK/INPUTS/<Target_Name>_pure_knitwork.csv
cp knitwork_impure_bulkdock_input.csv $BULK/INPUTS/<Target_Name>_impure_knitwork.csv
```


This uses the Fragmenstein placement function to place the scaffolds in both the inspiration A and inspiration B proteins

#### __In terminal:__

cd $BULK

```
python -m bulkdock place <Target Name> <Target_fragmenstein>.csv --split 2000

python -m bulkdock place <Target Name> <Target__knitwork>.csv --split 2000
```

#### Visualisers 

In [ ]:
from rdkit import Chem
import py3Dmol

sdf_path = "/opt/xchem-fragalysis-2/bemery/BulkDock/OUTPUTS/Zika_NS5_RdRp_fragmenstein_split2000_batch000_1179767.sdf"

suppl = Chem.SDMolSupplier(sdf_path, removeHs=False)
mols = [m for m in suppl if m is not None]

print(f"Loaded {len(mols)} ligands")

view = py3Dmol.view(width=1000, height=800)

for i, mol in enumerate(mols):
    block = Chem.MolToMolBlock(mol)
    view.addModel(block, "sdf")
    view.setStyle({"model": i}, {"stick": {"radius": 0.2}})

view.zoomTo()
view

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from rdkit import Chem
import py3Dmol

# Load molecules
suppl = Chem.SDMolSupplier(sdf_path, removeHs=False)
mols = [m for m in suppl if m is not None]

names = []
for i, m in enumerate(mols):
    if m.HasProp("_Name") and m.GetProp("_Name").strip():
        names.append(m.GetProp("_Name"))
    else:
        names.append(f"Ligand {i+1}")

print(f"Loaded {len(mols)} ligands")

# Widget UI
prev_btn = widgets.Button(description="◀ Previous")
next_btn = widgets.Button(description="Next ▶")
title = widgets.HTML()
out = widgets.Output()
state = {"i": 0}

def render(i):
    if not mols:
        with out:
            clear_output(wait=True)
            print("No molecules found in the SDF.")
        return

    i = i % len(mols)
    state["i"] = i

    with out:
        clear_output(wait=True)
        view = py3Dmol.view(width=1000, height=800)
        block = Chem.MolToMolBlock(mols[i])
        view.addModel(block, "sdf")
        view.setStyle({"stick": {"radius": 0.25}})
        view.zoomTo()
        display(view)

    title.value = f"<b>{i+1} / {len(mols)}</b> — {names[i]}"

def on_prev(_):
    render(state["i"] - 1)

def on_next(_):
    render(state["i"] + 1)

prev_btn.on_click(on_prev)
next_btn.on_click(on_next)

controls = widgets.HBox([prev_btn, next_btn, title])
display(controls, out)

render(0)

#### EXAMPLE ? Load outputted SDF to HIPPO

In [ ]:
# DONT NEED TO DO THIS, THIS WAS JUST AN EXAMPLE 

# BulkDock loads the outputted SDFs to HIPPO using this HIPPO command. For the purpose of training, I'm going to load a few
# example poses only

animal.load_sdf(
    target=target_name,
    path="/opt/xchem-fragalysis-2/bemery/BulkDock/OUTPUTS/Zika_NS5_RdRp_fragmenstein_split2000_batch000_1179767.sdf",
    compound_tags=["scaffold"],
    pose_tags=["fragmenstein"],
    name_col="ID",
    inspiration_col="inspiration_ids",
    reference_col="reference_id",
    
    ### Proposed extra arguments
    # enumeration_method = "fragmenstein", # to register the new compounds with the method used to conceive them
    # pose_method = "fragmenstein", # to register the new poses with the method used to calculate them
    # score_cols = ["energy_score", "distance_score"] # The SDF fields that correlate to the below scoring_methods
    # scoring_methods = [("fragmenstein_energy", "1.0.0"), ("fragmenstein_distance", "1.0.0")] # The pre-registered scoring_methods for the above score_cols
                                                                                           # along with the version number
)

# Output SDF of Scaffolds

In [ ]:
import molparse as mp
import hippo.pset as pset

pset.mp = mp

In [ ]:
poses = animal.poses.get_by_tag("Zika_NS5_RdRp_fragmenstein")
poses.to_fragalysis(
    "Zika_NS5_RdRp_bulkdock_poses.sdf", 
    method="fragmenstein", 
    submitter_name = "Ben Emery", 
    submitter_email= "ben.emery@cmd.ox.ac.uk", 
    submitter_institution="CMD", 
    copy_reference_pdbs=True
)

#### __optional - visualise compounds in trellis__

In [ ]:
from rdkit import Chem
from rdkit.Chem.Draw import MolsToGridImage
import ipywidgets as widgets
from IPython.display import display, clear_output

#should be done in previous step
#poses = animal.poses.get_by_tag("Zika_NS5_RdRp_fragmenstein") 
df = poses.get_df().copy()

# Keep one entry per compound rather than one per pose
for subset_col in ["compound_id", "HIPPO Compound ID", "inchikey"]:
    if subset_col in df.columns:
        df = df.drop_duplicates(subset=[subset_col])
        break

# Build RDKit mols from SMILES if needed
if "ROMol" not in df.columns:
    df["ROMol"] = df["smiles"].map(Chem.MolFromSmiles)

mols = [m for m in df["ROMol"] if m is not None]

per_page = 24
n_pages = max(1, (len(mols) + per_page - 1) // per_page)

page = widgets.IntSlider(
    value=0,
    min=0,
    max=n_pages - 1,
    step=1,
    description="Page",
    continuous_update=False,
    layout=widgets.Layout(width="500px"),
)

prev_btn = widgets.Button(description="◀ Previous")
next_btn = widgets.Button(description="Next ▶")
out = widgets.Output()

def render_page(p):
    start = p * per_page
    chunk = mols[start:start + per_page]
    img = MolsToGridImage(
        chunk,
        molsPerRow=6,
        subImgSize=(220, 180),
        legends=[""] * len(chunk),
    )
    with out:
        clear_output(wait=True)
        display(img)

def on_page_change(change):
    if change["name"] == "value":
        render_page(change["new"])

def go_prev(_):
    page.value = max(page.min, page.value - 1)

def go_next(_):
    page.value = min(page.max, page.value + 1)

page.observe(on_page_change)
prev_btn.on_click(go_prev)
next_btn.on_click(go_next)

display(widgets.HBox([prev_btn, next_btn, page]), out)
render_page(0)

# Rescore Hits with Gnina (To do: change GNINA path to path relative to XChem-FFF and add --gnina_path arg)

__In terminal:__

cd openbind-rescore

changes to make : input ligands sdf, input ref_pdb.zip
```
sbatch -p main -c 8 --wrap="singularity exec --nv --bind /opt/xchem-fragalysis-2:/opt/xchem-fragalysis-2 gnina/gnina_singularity.sif python -u gnina_rescore.py --input_ligands /opt/xchem-fragalysis-2/bemery/fffbe/Zika_NS5_RdRp_bulkdock_poses.sdf --ref_pdbs /opt/xchem-fragalysis-2/bemery/fffbe/Zika_NS5_RdRp_bulkdock_poses_refs.zip --output_path /opt/xchem-fragalysis-2/bemery/fffbe/gnina --cpu_cores 8"
```

In [ ]:
animal.poses

# Run MoCASSIn on Gnina Minimised Poses

Currently must be ran by Ron

# Reload New Scores to HIPPO

In [ ]:
# Note - this SDF contains GNINA minimised poses along with multiple GNINA and mocassin scores. 
# Currently, load_sdf expects an "energy_score_col" and a "distance_score_col" (e.g outputs from Fragmenstein placements)
# In the new schema, these arguments should be replaced with arguments that allow the user to specify which 
# fields are linked to which pre-registered scores. e.g

# Poses are in 1 sdf

animal.load_sdf(
    target= target_name,
    path="/opt/xchem-fragalysis-2/bemery/fffbe/gnina/output_sdfs/",
    compound_tags = ["scaffold"],
    pose_tags = ["gnina_repose"], # Note, this tag will not be necessary once the pose_method is implemented
    name_col = None,
    ### Proposed extra arguments
    # enumeration_method = None # No new compounds will be registered so user does not need to supply enumeration method
    # pose_method = "gnina_repose", # to register the new poses with the method used to calculate them
    # score_cols = ["CNN_VS", "moc_combo_multiref"] # The SDF fields that correlate to the below scoring_methods
    # scoring_methods = [("gnina_cnn_vs", "1.3.2"), ("moc_combo_multiref", "0.1.0")] # The pre-registered scoring_methods for the above score_cols
                                                                                 # along with the version number
)

In [ ]:
# Poses are in multiple sdfs

from pathlib import Path

sdf_dir = Path("/opt/xchem-fragalysis-2/bemery/fffbe/gnina/output_sdfs")

for sdf_file in sorted(sdf_dir.glob("*.sdf")):
    animal.load_sdf(
        target=target_name,
        path=sdf_file,
        compound_tags=["scaffold"],
        pose_tags=["gnina_repose"],
        name_col=None,
    )

In [ ]:
animal.poses(tag="gnina_repose")

# Scoring

Several possible ways to triage compounds here if there are too many to quote directly. 

Fragmenstein : 

Gnina : 

MoCASSIn : 

HIPPO : 



